In [0]:
%sql
CREATE TABLE workspace.prueba.nyctaxi_prueba
USING DELTA
AS
SELECT *
FROM csv.`/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz`

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

## Análisis inicial de la tabla Delta

La tabla fue creada en formato Delta, lo que permite utilizar funcionalidades avanzadas como Time Travel, OPTIMIZE y ZORDER.

### 🔹 Estado actual
- La tabla contiene **1 solo archivo (~100 MB)** (`numFiles = 1`)
- No tiene particiones ni clustering definidos

Esto indica que los datos ya están almacenados de forma eficiente y **no existe el problema de "small files"**.

### 🔹 Configuración y features
- Compresión: `ZSTD` (optimizada para rendimiento y tamaño)
- `deletionVectors` habilitado (permite borrar datos sin reescribir archivos)
- Versiones del protocolo:
  - Reader: 3
  - Writer: 7

Esto muestra que la tabla utiliza **features modernas de Delta Lake**.

### 🔹 Conclusión
La tabla se encuentra en un estado ya optimizado, por lo que para demostrar el impacto de OPTIMIZE y ZORDER será necesario simular un escenario con múltiples archivos pequeños.
 

## Simulación de ingesta incremental

Para recrear un escenario real, los datos se insertan en múltiples iteraciones usando pequeñas muestras del dataset.

Cada escritura agrega nuevos archivos a la tabla, generando fragmentación (small files).

Este patrón puede degradar el rendimiento y requiere optimización posterior.


In [0]:
df = spark.table("workspace.prueba.nyctaxi_prueba")
for i in range(20):
    df.sample(fraction=0.05).write.format("delta").mode("append").saveAsTable("workspace.prueba.nyctaxi_prueba")

#mode("append") no sobrescribe, agrega datos nuevos

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

##  Análisis luego de la ingesta incremental

Para simular un caso más realista, se realizó una **ingesta incremental** utilizando múltiples escrituras en modo `append`. Esto generó un escenario típico de producción donde los datos llegan en pequeños lotes.

### 🔹 Cambios en el estado de la tabla

- La tabla ahora contiene **43 archivos (~290 MB)** (`numFiles = 43`)
- Antes tenía **1 solo archivo (~100 MB)`
- No hay particiones ni clustering definidos

Esto evidencia la aparición del problema conocido como **"small files"**.

### 🔹 ¿Qué implica esto?

- Mayor cantidad de archivos → más operaciones de lectura (I/O)
- Incremento en el overhead del metadata
- Posible degradación en el rendimiento de las consultas

Este escenario es ideal para aplicar optimizaciones como:

- `OPTIMIZE` → compacta archivos pequeños en menos archivos grandes
- `ZORDER` → mejora la localización de datos en consultas con filtros

### 🔹 Configuración (sin cambios relevantes)

La tabla mantiene las mismas características modernas:

- Compresión: `ZSTD`
- `deletionVectors` habilitado
- Protocolos:
  - Reader: 3
  - Writer: 7

### 🔹 Conclusión

La tabla pasó de un estado **óptimo (1 archivo)** a un escenario más realista pero **subóptimo (43 archivos pequeños)** debido a la ingesta incremental.

Esto permite demostrar claramente el impacto de:

- `OPTIMIZE` → reducción de archivos y mejora en performance
- `ZORDER` → optimización para consultas selectivas

## 🔹 Paso 1: OPTIMIZE (compactación de archivos)

### 📊 Estado antes de OPTIMIZE

- Archivos: **43**
- Tamaño total: **~290 MB**
- Sin particiones ni clustering

Esto indica un escenario con **muchos archivos pequeños (small files)**.


In [0]:
%sql
optimize workspace.prueba.nyctaxi_prueba zorder by (_c1)

In [0]:
%sql
describe detaIl workspace.prueba.nyctaxi_prueba

## 📊 Comparación: antes vs después de OPTIMIZE

| Métrica        | Antes de OPTIMIZE | Después de OPTIMIZE |
|----------------|------------------|---------------------|
| Archivos       | 43               | 4                   |
| Tamaño total   | 290,641,208 bytes (~290 MB) | 294,454,740 bytes (~294 MB) |
| Última modificación | 2026-05-05 16:36:32 | 2026-05-05 17:15:24 |

---

### 🔍 Análisis

- La cantidad de archivos se redujo drásticamente (**43 → 4**)
- El tamaño total se mantiene prácticamente igual (ligeras variaciones por reorganización interna)
- Se eliminaron los *small files*, consolidando los datos en archivos más grandes

---

### ✅ Conclusión

`OPTIMIZE` logró:

- Mejorar la eficiencia de almacenamiento  
- Reducir el overhead de lectura  
- Preparar la tabla para consultas más rápidas  

---

## 🔹 ZORDER (optimización por columna)

Luego de ejecutar `OPTIMIZE`, se aplica `ZORDER` para mejorar el rendimiento de consultas con filtros.  
Ambas operaciones pueden ejecutarse en una misma instrucción.
## 💡 Buenas Prácticas para ZORDER

El **Z-Ordering** es una técnica de clustering que permite agrupar información relacionada en los mismos archivos. Para maximizar su impacto, sigue estas recomendaciones:

### ✅ Cuándo utilizar ZORDER
Se recomienda aplicar Z-Ordering en columnas que cumplan con los siguientes criterios:

*   **Columnas de alta cardinalidad** 
*   **Filtros frecuentes (`WHERE`)** 
*   **Consultas analíticas**
